# PCBert-Kla Research Workbook

This notebook documents and runs the cleaned PCBert-Kla replication, the proposed token-gated architecture, and the protein language model backbone comparison experiments.

The original PCBert-Kla training and independent test datasets are kept unchanged. The changes are made in the downstream model architecture, optimizer settings, and protein language model backbones.

## 1. Notebook Usage

Run this notebook from inside:

`experiment/pcbert_kla_clean/`

In Google Colab, after cloning the repository, use:

```python
%cd /content/Lysine-Lactylation-Sites-Prediction/experiment/pcbert_kla_clean
```

Heavy training cells are controlled by boolean flags such as `RUN_PROTBERT_TOKEN_GATED`. Keep them set to `False` until you intentionally want to run that experiment.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys

PROJECT_DIR = Path.cwd()
print("Current directory:", PROJECT_DIR)

expected = PROJECT_DIR / "scripts" / "run_replication.py"
if not expected.exists():
    raise RuntimeError(
        "This notebook should be run from experiment/pcbert_kla_clean. "
        f"Could not find {expected}."
    )

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project check passed.")

## 2. Dependency Setup

On Colab, set `INSTALL_DEPENDENCIES = True` once per fresh runtime. If dependencies are already installed, leave it as `False`.

In [ ]:
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        "requirements.txt",
    ])
else:
    print("Skipping dependency installation. Set INSTALL_DEPENDENCIES = True if needed.")

## 3. Helper Function for Reproducible Commands

The notebook uses the same command-line runner used in the project README. This keeps notebook runs aligned with the version-controlled implementation.

In [ ]:
def run_command(command: list[str], run: bool = True) -> None:
    printable = " ".join(shlex.quote(part) for part in command)
    print("$", printable)
    if run:
        subprocess.run(command, check=True)


def runner_args(*args: str) -> list[str]:
    return [sys.executable, "scripts/run_replication.py", *args]

## 4. Dataset Audit

This confirms that the original PCBert-Kla train/test files and physicochemical feature files are being used. It also prints the train/test overlap audit. The overlap is part of the original benchmark split; it is not introduced by this work.

In [ ]:
run_command(runner_args("--run", "data-check"), run=True)

## 5. Model Architecture Summary

The proposed method is the token-gated architecture. It keeps the same 51-amino-acid lysine-centered windows and 27 physicochemical features as PCBert-Kla, but changes the downstream neural framework.

Main components:

- protein language model token embeddings from ProtBert, ESM-2, ProtT5, or Ankh,
- central-lysine-guided token attention pooling,
- a separate physicochemical feature projection branch,
- gated fusion between sequence and physicochemical representations,
- residual classifier for final Kla probability prediction.

## 6. Reference Results Collected So Far

The following table records the main independent-test results obtained during the project. These are included for reporting and comparison. Re-run the corresponding command cells below if a fresh reproduction is needed.

In [ ]:
import pandas as pd

results = [
    {
        "Model": "Paper PCBert-Kla",
        "Setting": "reported independent test",
        "ACC": 0.9497,
        "AUC": 0.9646,
        "AUPRC": 0.9523,
        "F1": 0.9505,
        "MCC": 0.8999,
        "Pre": 0.9364,
        "Rec": 0.9650,
        "SP": 0.9345,
    },
    {
        "Model": "ProtBert token-gated",
        "Setting": "SGD, 4 encoder layers, fine-tuned",
        "ACC": 0.9632768362,
        "AUC": 0.9715439369,
        "AUPRC": 0.9585340424,
        "F1": 0.9635854342,
        "MCC": 0.9266867883,
        "Pre": 0.9555555556,
        "Rec": 0.9717514124,
        "SP": 0.9548022599,
    },
    {
        "Model": "Original architecture",
        "Setting": "AdamW, 3-seed ensemble",
        "ACC": 0.9604519774,
        "AUC": 0.9807686169,
        "AUPRC": 0.9627912005,
        "F1": 0.9606741573,
        "MCC": 0.9209627497,
        "Pre": 0.9553072626,
        "Rec": 0.9661016949,
        "SP": 0.9548022599,
    },
    {
        "Model": "ESM-2 35M token-gated",
        "Setting": "AdamW, 4 encoder layers, fine-tuned",
        "ACC": 0.9604519774,
        "AUC": 0.9719908072,
        "AUPRC": 0.9571143450,
        "F1": 0.9604519774,
        "MCC": 0.9209039548,
        "Pre": 0.9604519774,
        "Rec": 0.9604519774,
        "SP": 0.9604519774,
    },
    {
        "Model": "Ankh-base token-gated",
        "Setting": "AdamW, 4 encoder layers, frozen",
        "ACC": 0.9604519774,
        "AUC": 0.9745603115,
        "AUPRC": 0.9699405723,
        "F1": 0.9606741573,
        "MCC": 0.9209627497,
        "Pre": 0.9553072626,
        "Rec": 0.9661016949,
        "SP": 0.9548022599,
    },
    {
        "Model": "ProtT5 token-gated",
        "Setting": "AdamW, 4 encoder layers, frozen",
        "ACC": 0.9406779661,
        "AUC": 0.9714481790,
        "AUPRC": 0.9702612039,
        "F1": 0.9418282548,
        "MCC": 0.8820459824,
        "Pre": 0.9239130435,
        "Rec": 0.9604519774,
        "SP": 0.9209039548,
    },
]

results_df = pd.DataFrame(results)
results_df

## 7. Difference from the Paper Baseline

This computes metric differences relative to the reported PCBert-Kla independent-test results.

In [ ]:
metric_cols = ["ACC", "AUC", "AUPRC", "F1", "MCC", "Pre", "Rec", "SP"]
paper = results_df.loc[results_df["Model"] == "Paper PCBert-Kla", metric_cols].iloc[0]

diff_df = results_df.copy()
for col in metric_cols:
    diff_df[col] = diff_df[col] - paper[col]

diff_df[diff_df["Model"] != "Paper PCBert-Kla"][["Model", "Setting", *metric_cols]]

## 8. Plot the Main Comparison

This creates a compact visualization for key independent-test metrics. It is intended for internal inspection, not necessarily as a final paper figure.

In [ ]:
import matplotlib.pyplot as plt

plot_cols = ["ACC", "F1", "MCC", "AUC", "AUPRC"]
plot_df = results_df.set_index("Model")[plot_cols]

ax = plot_df.plot(kind="bar", figsize=(14, 5), ylim=(0.85, 1.0))
ax.set_title("Independent-Test Performance Comparison")
ax.set_ylabel("Score")
ax.legend(loc="lower right")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 9. Baseline Replication Commands

These cells reproduce the original-style baseline. They are heavy training runs, so they are disabled by default.

In [ ]:
RUN_BASELINE_INDEPENDENT = False

baseline_independent_cmd = runner_args(
    "--run", "independent",
    "--epochs", "30",
    "--batch-size", "4",
    "--device", "cuda",
)

run_command(baseline_independent_cmd, run=RUN_BASELINE_INDEPENDENT)

In [ ]:
RUN_BASELINE_CV = False

baseline_cv_cmd = runner_args(
    "--run", "cv",
    "--epochs", "30",
    "--batch-size", "4",
    "--device", "cuda",
)

run_command(baseline_cv_cmd, run=RUN_BASELINE_CV)

## 10. Proposed Main Model: ProtBert + Token-Gated + SGD

This is the strongest single-model architecture result obtained so far. It fine-tunes a truncated 4-layer ProtBert encoder together with the token-gated prediction head.

In [ ]:
RUN_PROTBERT_TOKEN_GATED = False

protbert_token_gated_cmd = runner_args(
    "--run", "independent",
    "--architecture", "token_gated",
    "--epochs", "30",
    "--batch-size", "4",
    "--device", "cuda",
    "--optimizer", "sgd",
    "--learning-rate", "0.003",
    "--weight-decay", "0.0",
    "--scheduler", "none",
)

run_command(protbert_token_gated_cmd, run=RUN_PROTBERT_TOKEN_GATED)

## 11. Backbone Comparison Experiments

The following commands test whether the token-gated framework transfers across protein language model backbones. Run one at a time on Colab GPU.

In [ ]:
backbone_commands = {
    "ESM-2 35M token-gated AdamW": runner_args(
        "--run", "independent",
        "--architecture", "token_gated",
        "--model-name", "facebook/esm2_t12_35M_UR50D",
        "--epochs", "30",
        "--batch-size", "4",
        "--device", "cuda",
        "--optimizer", "adamw",
        "--learning-rate", "2e-5",
        "--weight-decay", "0.01",
        "--scheduler", "linear",
        "--warmup-ratio", "0.1",
    ),
    "ProtT5 token-gated frozen": runner_args(
        "--run", "independent",
        "--architecture", "token_gated",
        "--model-name", "Rostlab/prot_t5_xl_half_uniref50-enc",
        "--encoder-layers", "4",
        "--freeze-encoder",
        "--epochs", "30",
        "--batch-size", "2",
        "--device", "cuda",
        "--optimizer", "adamw",
        "--learning-rate", "1e-4",
        "--weight-decay", "0.01",
        "--scheduler", "linear",
        "--warmup-ratio", "0.1",
    ),
    "Ankh-base token-gated frozen": runner_args(
        "--run", "independent",
        "--architecture", "token_gated",
        "--model-name", "ElnaggarLab/ankh-base",
        "--encoder-layers", "4",
        "--freeze-encoder",
        "--epochs", "30",
        "--batch-size", "2",
        "--device", "cuda",
        "--optimizer", "adamw",
        "--learning-rate", "1e-4",
        "--weight-decay", "0.01",
        "--scheduler", "linear",
        "--warmup-ratio", "0.1",
    ),
}

for name, command in backbone_commands.items():
    print("\n#", name)
    print(" ".join(shlex.quote(part) for part in command))

In [ ]:
RUN_BACKBONE_NAME = None
# Example:
# RUN_BACKBONE_NAME = "Ankh-base token-gated frozen"

if RUN_BACKBONE_NAME is not None:
    run_command(backbone_commands[RUN_BACKBONE_NAME], run=True)
else:
    print("No backbone command selected. Set RUN_BACKBONE_NAME to one of:")
    for key in backbone_commands:
        print("-", key)

## 12. Ensemble Experiment

Ensembling trains multiple models with different random seeds and averages their independent-test probabilities. It is computationally heavier than single-seed runs.

In [ ]:
RUN_PROTBERT_TOKEN_GATED_ENSEMBLE = False

protbert_token_gated_ensemble_cmd = runner_args(
    "--run", "ensemble-independent",
    "--architecture", "token_gated",
    "--epochs", "30",
    "--batch-size", "4",
    "--device", "cuda",
    "--optimizer", "sgd",
    "--learning-rate", "0.003",
    "--weight-decay", "0.0",
    "--scheduler", "none",
    "--ensemble-seeds", "42,123,2025",
)

run_command(protbert_token_gated_ensemble_cmd, run=RUN_PROTBERT_TOKEN_GATED_ENSEMBLE)

## 13. Saved Outputs

Independent and ensemble runs save prediction CSV files under `outputs/` by default. Use this cell to inspect generated files after running experiments.

In [ ]:
outputs_dir = PROJECT_DIR / "outputs"
if outputs_dir.exists():
    for path in sorted(outputs_dir.rglob("*")):
        if path.is_file():
            print(path.relative_to(PROJECT_DIR))
else:
    print("No outputs directory yet. Run an experiment first.")

## 14. Reproducibility Notes

- The training and independent test files are the same files used by the PCBert-Kla baseline repository.
- No additional samples are added to the benchmark.
- ProtBert and ESM-2 experiments fine-tune truncated encoder layers unless `--freeze-encoder` is provided.
- ProtT5 and Ankh experiments above use frozen encoder settings for Colab stability and as backbone-ablation entries.